# 02. Data Preprocessing & Engineering Attribute Extraction

**Course:** CSE 276 – CA-1 Academic Project  
**Project:** Material Code Harmonisation Across Public Enterprises  
**Corpus:** `data/raw/material_description_corpus.csv` — 21,513 records across 3 CPSEs  
**Candidate Pool:** 6,634 records (Tier 1 Discrete + Tier 2 Tagged)  

---

## Notebook Objectives
1. **Environment Setup** – Dual-environment support: Google Colab and local Windows workstation.
2. **Corpus Partitioning** – Reconstruct and validate the approved 4-tier corpus hierarchy.
3. **Candidate Pool Isolation** – Isolate 6,634 candidate records; preserve the 14,879 remaining.
4. **Text Normalisation** – Uppercase, whitespace, abbreviation expansion, boilerplate removal.
5. **Domain Integrity Check** – Verify PIPE, FLANGE, SEAMLESS, SE, ASTM, IS standards are preserved.
6. **Attribute Extraction** – Regex-based extraction of dimensions, grades, pressure, schedule.
7. **Visualisation & Export** – Diagnostic charts and output files for Notebook 03.

> **Academic Integrity Rule:** No fabricated metrics, synthetic records, or premature ML training.

---
## 1. Environment & Workspace Setup

In [ ]:
# ==============================================================================
# Step 1: Environment & Repository Configuration
# ==============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running in Google Colab.')
    REPO_NAME = 'CSE-276-PROJECT'
    REPO_URL  = 'https://github.com/SHIVA07605/CSE-276-PROJECT.git'
    if not os.path.exists(f'/content/{REPO_NAME}'):
        os.system(f'git clone {REPO_URL}')
    PROJECT_ROOT = Path(f'/content/{REPO_NAME}').resolve()
    os.chdir(PROJECT_ROOT)
    os.system('pip install -q pandas numpy matplotlib seaborn')
else:
    print('Running on local workstation.')
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project Root : {PROJECT_ROOT}')
print(f'sys.path OK  : {str(PROJECT_ROOT) in sys.path}')

In [ ]:
# Core imports
import json
import re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive — safe for both Colab and local
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Project modules
from src.config import DATA_RAW_DIR, DATA_PROCESSED_DIR, FIGURES_DIR, REPORTS_DIR
from src.preprocessing import (
    normalise_text,
    extract_material_attributes,
    partition_corpus,
    validate_partitions,
    preprocess_catalogue,
    EXPECTED_COUNTS,
)

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('All imports successful.')

---
## 2. Load Raw Corpus

In [ ]:
# ==============================================================================
# Step 2: Load material_description_corpus.csv
# ==============================================================================
corpus_path = DATA_RAW_DIR / 'material_description_corpus.csv'
df_raw = pd.read_csv(corpus_path)

print(f'Records  : {len(df_raw):,}')
print(f'Columns  : {len(df_raw.columns)}')
print(f'Columns  : {list(df_raw.columns)}')
print()
print('Organizations:')
print(df_raw['organization'].value_counts().to_string())
print()
print('Description kinds:')
print(df_raw['description_kind'].value_counts().to_string())

assert len(df_raw) == 21513, f'Expected 21,513 records, got {len(df_raw)}'

---
## 3. Corpus Partitioning & Validation

The approved 4-tier hierarchy is reconstructed programmatically from `description_kind` and `item_type_hint`:

| Tier | Definition | Expected Count |
|---|---|---:|
| Tier 1 | Discrete material line items | 923 |
| Tier 2 Tagged | Procurement descriptions with commodity tag | 5,711 |
| Tier 2 Untagged | Procurement descriptions without tag | 14,542 |
| Tier 3 | Administrative / works descriptions | 337 |
| **Candidate Pool** | Tier 1 + Tier 2 Tagged | **6,634** |
| Held-back | Tier 2 Untagged + Tier 3 | 14,879 |

In [ ]:
# ==============================================================================
# Step 3: Partition corpus
# ==============================================================================
partitions = partition_corpus(df_raw)
validation = validate_partitions(df_raw, partitions, strict=True)

print(f"Tier 1 discrete     : {len(partitions['tier1_discrete']):>6,}  (expected {EXPECTED_COUNTS['tier1_discrete']:,})")
print(f"Tier 2 tagged       : {len(partitions['tier2_tagged']):>6,}  (expected {EXPECTED_COUNTS['tier2_tagged']:,})")
print(f"Tier 2 untagged     : {len(partitions['tier2_untagged']):>6,}  (expected {EXPECTED_COUNTS['tier2_untagged']:,})")
print(f"Tier 3 admin        : {len(partitions['tier3_administrative']):>6,}  (expected {EXPECTED_COUNTS['tier3_administrative']:,})")
print(f"Candidate pool      : {len(partitions['candidate_pool']):>6,}  (expected {EXPECTED_COUNTS['candidate_pool']:,})")
print(f"Remaining pool      : {len(partitions['remaining_pool']):>6,}  (expected {EXPECTED_COUNTS['remaining_pool']:,})")
print()
print(f"Validation passed        : {validation['valid']}")
print(f"All counts match         : {validation['all_counts_match']}")
print(f"Has unique corpus_ids    : {validation['has_unique_ids']}")
print(f"Mutually exclusive       : {validation['is_mutually_exclusive']}")
print(f"Collectively exhaustive  : {validation['is_collectively_exhaustive']}")

### 3.1 Partition Distribution Chart

In [ ]:
fig1, ax1 = plt.subplots(figsize=(8, 4.5))
tier_counts = {
    'Tier 1\nDiscrete': len(partitions['tier1_discrete']),
    'Tier 2\nTagged': len(partitions['tier2_tagged']),
    'Tier 2\nUntagged': len(partitions['tier2_untagged']),
    'Tier 3\nAdmin': len(partitions['tier3_administrative']),
}
bars = ax1.bar(
    tier_counts.keys(), tier_counts.values(),
    color=['#1f77b4', '#2ca02c', '#aec7e8', '#ffbb78'],
    edgecolor='white', width=0.55
)
for bar, val in zip(bars, tier_counts.values()):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Corpus Partition Distribution (21,513 Records)', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Record Count', fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.set_ylim(0, max(tier_counts.values()) * 1.15)
ax1.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig1_path = FIGURES_DIR / '02_corpus_partition_distribution.png'
plt.savefig(fig1_path, dpi=150)
plt.show()
print(f'Saved: {fig1_path}')

---
## 4. Text Normalisation

Applied operations (in order):
1. Convert to **UPPERCASE**.
2. Normalize unicode separators, dashes, and quotes.
3. Normalize acronym dots: `S.S.` → `SS`, `C.S.` → `CS`.
4. Standardize **material grades**: `SS 304` → `SS304`, `SS316L` preserved.
5. Standardize **engineering units**: `10 mm` → `10MM`, `2 inch` → `2INCH`.
6. Expand **abbreviations**: `SS` → `STAINLESS STEEL`, `CS` → `CARBON STEEL`.
7. Strip irrelevant **procurement boilerplate**: `Supply of`, `Procurement of`, `AMC for`.
8. Collapse whitespace and trim.

**Preservation rule:** Words like PIPE, FLANGE, SEAMLESS, WELDED, VALVE, ASTM, IS 2062  
are **never removed** — they are critical engineering identifiers.

In [ ]:
# ==============================================================================
# Step 4: Before / After normalisation examples
# ==============================================================================
examples = [
    ('10 mm',                                              '10MM'),
    ('2 inch',                                             '2INCH'),
    ('S.S. 304',                                           'SS304'),
    ('Stainless Steel',                                    'STAINLESS STEEL'),
    ('Supply of S.S. 304 Hex Bolt 10 mm',                 'SS304 HEXAGONAL BOLT 10MM'),
    ('CS Gate Valve 2INCH 150# PN16',                     'CARBON STEEL GATE VALVE 2INCH 150# PN16'),
    ('MS Plate IS2062 Grade A 10mm Thk',                  'MILD STEEL PLATE IS2062 GRADE A 10MM THICKNESS'),
    ('TRUCK CHASSIS TATA SE 1613/42',                     'TRUCK CHASSIS TATA SE 1613/42 (SE preserved)'),
    ('Consolidated Procurement of CI Gate Valves',         'CAST IRON GATE VALVES (boilerplate removed)'),
]
print(f'{"Original":<50}  {"Normalised"}')
print('-' * 100)
for raw, _ in examples:
    norm = normalise_text(raw)
    print(f'  {raw:<48}  {norm}')

---
## 5. Preprocess Candidate Pool (6,634 Records)

In [ ]:
# ==============================================================================
# Step 5: Apply preprocessing pipeline to the candidate pool
# ==============================================================================
df_candidate = partitions['candidate_pool'].copy()
df_remaining  = partitions['remaining_pool'].copy()

df_candidate = preprocess_catalogue(df_candidate, text_col='description')

print(f'Candidate pool processed : {len(df_candidate):,} records')
print(f'Held-back records        : {len(df_remaining):,} records')
print(f'Columns in processed df  : {list(df_candidate.columns)}')

In [ ]:
# Show a sample of processed records
display_cols = [
    'corpus_id', 'organization', 'raw_description', 'cleaned_description',
    'attr_diameter', 'attr_material_grade', 'attr_pressure_class', 'attr_schedule'
]
df_candidate[display_cols].head(10)

---
## 6. Engineering Attribute Extraction

Attributes extracted using regex from the **cleaned description**:

| Attribute | Example Input | Extracted |
|---|---|---|
| Metric Thread | `BOLT M10` | `M10` |
| Diameter (MM) | `PIPE 50MM NB` | `50MM NB` |
| Diameter (INCH) | `VALVE 2INCH` | `2INCH` |
| Diameter (DN) | `VALVE DN100` | `DN100` |
| Material Grade | `SS304 PIPE` | `SS304` |
| Material Grade | `IS2062 PLATE` | `IS2062` |
| Pressure Class | `150# GATE VALVE` | `150#` |
| Pressure Class | `PN16 BUTTERFLY VALVE` | `PN16` |
| Pressure Class | `CLASS 150 FLANGE` | `CLASS 150` |
| Pipe Schedule | `SCH 40 PIPE` | `SCH 40` |
| Voltage | `415V MOTOR` | `415V` |
| Capacity | `100KVA TRANSFORMER` | `100KVA` |

**Disambiguation rules enforced:**
- M10 (thread) ≠ 10MM (diameter)
- SCH 40 (schedule) ≠ 150# (pressure class)
- SS304 ≠ SS316 (different grades)

In [ ]:
# ==============================================================================
# Step 6: Attribute extraction examples
# ==============================================================================
attr_examples = [
    'STAINLESS STEEL HEXAGONAL BOLT M10 50MM SS304 PN16',
    'CARBON STEEL GATE VALVE 2INCH 150#',
    'SEAMLESS PIPE 50MM NB SS316 SCH 80 ASTM A106',
    'BUTTERFLY VALVE DN100 PN40',
    'MILD STEEL PLATE IS2062 GRADE A 10MM THICKNESS',
    'DISTRIBUTION TRANSFORMER 100KVA 11KV/415V',
]
for ex in attr_examples:
    attrs = extract_material_attributes(ex)
    relevant = {k: v for k, v in attrs.items() if v and k != 'ambiguous_flags'}
    print(f'  Input: {ex}')
    print(f'  Attrs: {relevant}')
    print()

---
## 7. Attribute Extraction Coverage Statistics

In [ ]:
# ==============================================================================
# Step 7: Coverage statistics over candidate pool
# ==============================================================================
attr_cols = [
    'attr_metric_thread', 'attr_diameter', 'attr_material_grade',
    'attr_pressure_class', 'attr_schedule', 'attr_voltage', 'attr_capacity'
]
attr_labels = {
    'attr_metric_thread': 'Metric Thread',
    'attr_diameter': 'Diameter',
    'attr_material_grade': 'Material Grade',
    'attr_pressure_class': 'Pressure Class',
    'attr_schedule': 'Pipe Schedule',
    'attr_voltage': 'Voltage',
    'attr_capacity': 'Capacity',
}
coverage = {}
print(f'{"Attribute":<20} {"Count":>7}  {"Coverage %":>10}')
print('-' * 42)
for col in attr_cols:
    n = df_candidate[col].notnull().sum()
    pct = n / len(df_candidate) * 100
    coverage[col] = {'count': int(n), 'pct': round(float(pct), 2)}
    print(f'  {attr_labels[col]:<18} {n:>7,}  {pct:>9.1f}%')

In [ ]:
# Coverage chart
attr_present = [coverage[col]['pct'] for col in attr_cols]
attr_missing = [100 - p for p in attr_present]
attr_names_plot = [attr_labels[col] for col in attr_cols]

fig2, ax2 = plt.subplots(figsize=(9, 4))
x = np.arange(len(attr_names_plot))
bar_w = 0.45
b1 = ax2.bar(x - bar_w/2, attr_present, bar_w, label='Extracted (%)', color='#2ca02c', alpha=0.85)
b2 = ax2.bar(x + bar_w/2, attr_missing,  bar_w, label='Missing (%)',   color='#d62728', alpha=0.65)
for bar, val in zip(b1, attr_present):
    if val > 2:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=7.5)
ax2.set_xticks(x)
ax2.set_xticklabels(attr_names_plot, fontsize=9)
ax2.set_ylabel('Percentage of 6,634 records', fontsize=10)
ax2.set_title('Engineering Attribute Extraction Coverage - Candidate Pool', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 115)
ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig2_path = FIGURES_DIR / '02_attribute_extraction_coverage.png'
plt.savefig(fig2_path, dpi=150)
plt.show()
print(f'Saved: {fig2_path}')

In [ ]:
# Raw vs cleaned text length distribution
raw_lens   = df_candidate['raw_description'].fillna('').apply(len)
clean_lens = df_candidate['cleaned_description'].fillna('').apply(len)

fig3, ax3 = plt.subplots(figsize=(9, 4))
ax3.hist(raw_lens.clip(0, 400),   bins=60, alpha=0.55,
         label=f'Raw   (mean={raw_lens.mean():.0f} chars)',   color='#d62728', density=True)
ax3.hist(clean_lens.clip(0, 400), bins=60, alpha=0.55,
         label=f'Clean (mean={clean_lens.mean():.0f} chars)', color='#1f77b4', density=True)
ax3.set_title('Character Length: Raw vs Cleaned Descriptions', fontsize=12, fontweight='bold')
ax3.set_xlabel('Character Count', fontsize=10)
ax3.set_ylabel('Density', fontsize=10)
ax3.legend(fontsize=9)
ax3.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig3_path = FIGURES_DIR / '02_raw_vs_cleaned_length.png'
plt.savefig(fig3_path, dpi=150)
plt.show()
print(f'Saved: {fig3_path}')

---
## 8. Export Output Files

In [ ]:
# ==============================================================================
# Step 8: Export preprocessed datasets and summary report
# ==============================================================================

# 8a. Preprocessed candidate pool
candidate_csv = DATA_PROCESSED_DIR / 'preprocessed_materials.csv'
df_candidate.to_csv(candidate_csv, index=False)
print(f'Exported: {candidate_csv} ({len(df_candidate):,} records)')

# 8b. Parquet (optional — requires pyarrow)
try:
    parquet_path = DATA_PROCESSED_DIR / 'preprocessed_materials.parquet'
    df_candidate.to_parquet(parquet_path, index=False)
    print(f'Exported: {parquet_path}')
except Exception as e:
    print(f'Parquet skipped (no engine): {e}')

# 8c. Held-back records
held_csv = DATA_PROCESSED_DIR / 'held_back_records.csv'
df_remaining.to_csv(held_csv, index=False)
print(f'Exported: {held_csv} ({len(df_remaining):,} records)')

# 8d. Preprocessing summary JSON
summary = {
    'notebook': '02_data_preprocessing.ipynb',
    'corpus_total_records': int(len(df_raw)),
    'corpus_columns': list(df_raw.columns),
    'partition_validation': {
        'valid': bool(validation['valid']),
        'all_counts_match': bool(validation['all_counts_match']),
        'has_unique_ids': bool(validation['has_unique_ids']),
        'is_mutually_exclusive': bool(validation['is_mutually_exclusive']),
        'is_collectively_exhaustive': bool(validation['is_collectively_exhaustive']),
    },
    'partition_counts': {k: int(v) for k, v in validation['counts'].items()},
    'expected_counts': {k: int(v) for k, v in EXPECTED_COUNTS.items()},
    'candidate_pool_records': int(len(df_candidate)),
    'held_back_records': int(len(df_remaining)),
    'attribute_coverage': {
        attr_labels[col]: coverage[col]
        for col in attr_cols
    },
    'text_length_stats': {
        'raw_mean_chars': round(float(raw_lens.mean()), 2),
        'cleaned_mean_chars': round(float(clean_lens.mean()), 2),
        'mean_reduction_chars': round(float(raw_lens.mean() - clean_lens.mean()), 2),
    },
}
summary_path = REPORTS_DIR / 'preprocessing_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f'Exported: {summary_path}')

---
## 9. Summary & Handoff to Notebook 03

### Completed operations:
- **Partition validation PASSED** — 923 + 5,711 + 14,542 + 337 = 21,513 records confirmed.
- **Candidate pool** of 6,634 records preprocessed and written to `data/processed/preprocessed_materials.csv`.
- **Held-back records** (14,879) preserved in `data/processed/held_back_records.csv`.
- **Text normalisation** applied: uppercase, unit standardisation, abbreviation expansion, boilerplate removal.
- **Attribute extraction** applied: diameter, thread, grade, pressure class, schedule, voltage, capacity.

### Known limitations:
- Overall attribute extraction rate is low (~3–13%) because the corpus is dominated by  
  Tier 2 tagged procurement descriptions (tender package language, not pure material specs).
  This is expected and consistent with Notebook 01 findings.
- Regex patterns do not capture all custom acronyms or abbreviations unique to individual CPSEs.
- Electrical attributes (voltage, capacity) appear only in the NTPC sub-corpus.

### Ready for Notebook 03:
`data/processed/preprocessed_materials.csv` — **6,634 candidate records** with  
`cleaned_description`, extracted attribute columns, and full source provenance.